Changing my current work directory to project root , so i can get the db without actually changing the code in load.

In [85]:
import os
from pathlib import Path

# Find project root directory (directory containing raw/ or config.py)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change current working directory to project root
os.chdir(PROJECT_ROOT)

print(f"Working directory set to project root: {os.getcwd()}")

Working directory set to project root: c:\Users\Likhita\Desktop\pokemon-etl


# Database Walkthrough

The goal is to understand **how our transformed data is finally stored and retrieved from the database**.

In [86]:
import sqlite3
import pandas as pd

## 1. Connecting to the Database

SQLite stores the complete database in a single `.db` file.

We use Python's `sqlite3` library to connect to that file.

In [87]:
DB_PATH = "pokemon.db"

conn = sqlite3.connect(DB_PATH)

print("Connected to SQLite database successfully!")

Connected to SQLite database successfully!


## Our Database Tables

Our database contains **6 tables**:

| Table | Purpose |
|---|---|
| `pokemon` | Main Pokémon information and calculated statistics |
| `pokemon_types` | Types associated with each Pokémon |
| `pokemon_abilities` | Abilities associated with each Pokémon |
| `pokemon_moves` | Moves that each Pokémon can learn |
| `moves` | Details about each move |
| `pokemon_species` | Species-level information about each Pokémon |

The `pokemon` table acts as the main table.

The other tables except pokemon-species , store additional information related to a Pokémon.

In [88]:
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

tables

,name
0,moves
1,pokemon
2,pokemon_abilities
3,pokemon_moves
4,pokemon_species
5,pokemon_types


In [89]:
tables = pd.read_sql_query("""
    SELECT COUNT(*) FROM pokemon;
""", conn)

tables

,COUNT(*)
0,1351


In [90]:
tables = pd.read_sql_query("""
    SELECT pokemon_id, COUNT(*)
FROM pokemon
GROUP BY pokemon_id
HAVING COUNT(*) > 1;
""", conn)

tables

,pokemon_id,COUNT(*)


                         ┌──────────────────────┐
                         │   pokemon_species    │
                         │----------------------│
                         │ species_id (PK)      │
                         │ generation           │
                         │ color                │
                         │ habitat              │
                         │ special_status       │
                         └──────────|───────────┘
                                    │
                                    │ species_id
                                    │
                         ┌──────────▼───────────┐
                         │       pokemon        │
                         │----------------------│
                         │ pokemon_id (PK)      │
                         │ species_id (FK)      │
                         │ pokemon_name         │
                         │ stats                │
                         │ derived attributes   │
                         └──────┬─────┬─────────┘
                                │     │
                 ┌──────────────┘     └──────────────┐
                 │                                   │
                 ▼                                   ▼
       ┌──────────────────┐                ┌──────────────────┐
       │  pokemon_types   │                │ pokemon_abilities│
       │------------------│                │------------------│
       │ pokemon_id (FK)  │                │ pokemon_id (FK)  │
       │ type_name        │                │ ability_name     │
       │ slot             │                │ slot             │
       └──────────────────┘                │ is_hidden        │
                                           └──────────────────┘

                         pokemon
                            │
                            │
                            ▼
                   ┌──────────────────┐
                   │  pokemon_moves   │
                   │------------------│
                   │ pokemon_id (FK)  │
                   │ move_id (FK)     │
                   │ learn_method     │
                   │ level_learned_at │
                   │ version_group    │
                   └────────┬─────────┘
                            │
                            │ move_id
                            ▼
                   ┌──────────────────┐
                   │      moves       │
                   │------------------│
                   │ move_id (PK)     │
                   │ move_name        │
                   │ move_type        │
                   │ power            │
                   │ accuracy         │
                   │ damage_class     │
                   └──────────────────┘

## `pokemon` Table

This is our **main Pokémon table**.

It contains information that directly describes a Pokémon.

### Important columns

- `pokemon_id` → unique ID of the Pokémon
- `species_id` → identifies the species
- `pokemon_name` → Pokémon name
- `form_name`→ form name
- `is_default_form` → whether it is the default form
- `height_m`, `weight_kg` → physical measurements
- `base_experience` → base experience value
- `hp`, `attack`, `defense` → basic stats
- `special_attack`, `special_defense`, `speed` → remaining battle stats
- `total_base_stats` → sum of the six base stats
- `offensive_power` → calculated offensive measure
- `defensive_power` → calculated defensive measure
- `speed_percentile` → position of the Pokémon's speed compared with others
- `battle_style` → derived classification
- `stat_specialization` → identifies the Pokémon's statistical specialization
- `size_class` → derived size classification

So this table contains both **source information and values derived during transformation**.

In [91]:
pokemon_df = pd.read_sql_query("""
    SELECT *
    FROM pokemon
    LIMIT 5;
""", conn)

pokemon_df

,pokemon_id,species_id,name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,...,special_attack,special_defense,speed,total_base_stats,offensive_power,defensive_power,speed_percentile,battle_style,stat_specialization,size_class
0,1,1,bulbasaur,bulbasaur,1,0.7,6.9,64,45,49,...,65,65,45,318,114,114,22.02,Balanced,special_attack,Small
1,2,2,ivysaur,ivysaur,1,1.0,13.0,142,60,62,...,80,80,60,405,142,143,37.93,Balanced,special_attack,Medium
2,3,3,venusaur,venusaur,1,2.0,100.0,236,80,82,...,100,100,80,525,182,183,59.99,Balanced,special_attack,Large
3,4,4,charmander,charmander,1,0.6,8.5,62,39,52,...,60,50,65,309,112,93,43.89,Balanced,speed,Small
4,5,5,charmeleon,charmeleon,1,1.1,19.0,142,58,64,...,80,65,80,405,144,123,59.99,Offensive,special_attack,Medium


## `pokemon_types` Table

A Pokémon can have **one or two types**.

Instead of storing multiple types inside one column of the `pokemon` table, we store them as separate rows.

For example:

```text
pokemon_id | type_name | slot
25         | electric  | 1
```

For a dual-type Pokémon:

```text
pokemon_id | type_name | slot
1          | grass     | 1
1          | poison    | 2
```

### Columns

- `pokemon_id` → identifies the Pokémon
- `type_name` → Pokémon's type
- `slot` → position of the type

This keeps the data structured and avoids storing values such as `"grass, poison"` in a single column.

In [92]:
pokemon_types_df = pd.read_sql_query("""
    SELECT *
    FROM pokemon_types
    LIMIT 10;
""", conn)

pokemon_types_df

,pokemon_id,type_name,slot
0,1,grass,1
1,1,poison,2
2,2,grass,1
3,2,poison,2
4,3,grass,1
5,3,poison,2
6,4,fire,1
7,5,fire,1
8,6,fire,1
9,6,flying,2


## `pokemon_abilities` Table

A Pokémon can have multiple abilities, so abilities are stored separately from the main Pokémon information.

### Columns

- `pokemon_id` → identifies the Pokémon
- `ability_name` → name of the ability
- `slot` → ability position
- `is_hidden` → identifies whether the ability is a hidden ability

The primary key is:

```text
pokemon_id + slot
```

This ensures that the same Pokémon cannot have two records with the same ability slot.

In [93]:
pokemon_abilities_df = pd.read_sql_query("""
    SELECT *
    FROM pokemon_abilities
    LIMIT 10;
""", conn)

pokemon_abilities_df

,pokemon_id,ability_name,slot,is_hidden
0,1,overgrow,1,0
1,1,chlorophyll,3,1
2,2,overgrow,1,0
3,2,chlorophyll,3,1
4,3,overgrow,1,0
5,3,chlorophyll,3,1
6,4,blaze,1,0
7,4,solar-power,3,1
8,5,blaze,1,0
9,5,solar-power,3,1


## `moves` Table

This table stores information about the **moves themselves**.

### Columns

- `move_id` → unique ID of the move
- `move_name` → name of the move
- `move_type` → type of the move
- `power` → move power
- `accuracy` → accuracy percentage
- `pp` → number of times the move can be used
- `priority` → move priority
- `damage_class` → physical, special, or status
- `generation` → generation in which the move was introduced

This table describes the move once, rather than repeating its details for every Pokémon that can learn it.

In [94]:
moves_df = pd.read_sql_query("""
    SELECT *
    FROM moves
    LIMIT 10;
""", conn)

moves_df

,move_id,move_name,move_type,power,accuracy,pp,priority,damage_class,generation
0,1,pound,normal,40,100,35,0,physical,generation-i
1,2,karate-chop,fighting,50,100,25,0,physical,generation-i
2,3,double-slap,normal,15,85,10,0,physical,generation-i
3,4,comet-punch,normal,18,85,15,0,physical,generation-i
4,5,mega-punch,normal,80,85,20,0,physical,generation-i
5,6,pay-day,normal,40,100,20,0,physical,generation-i
6,7,fire-punch,fire,75,100,15,0,physical,generation-i
7,8,ice-punch,ice,75,100,15,0,physical,generation-i
8,9,thunder-punch,electric,75,100,15,0,physical,generation-i
9,10,scratch,normal,40,100,35,0,physical,generation-i


## `pokemon_moves` Table

This table connects **Pokémon and moves**.

A Pokémon can learn many moves, and one move can be learned by many Pokémon.

So this table works as a **relationship table**.

### Columns

- `pokemon_id` → Pokémon
- `move_id` → move
- `learn_method` → how the Pokémon learns the move
- `level_learned_at` → level at which it is learned
- `version_group` → game/version group

The primary key contains all five columns, because the same Pokémon and move can have different learning conditions.

In [95]:
pokemon_moves_df = pd.read_sql_query("""
    SELECT *
    FROM pokemon_moves
    LIMIT 10;
""", conn)

pokemon_moves_df

,pokemon_id,move_id,learn_method,level_learned_at,version_group
0,1,13,egg,0,gold-silver
1,1,13,egg,0,crystal
2,1,14,machine,0,red-blue
3,1,14,machine,0,yellow
4,1,14,tutor,0,emerald
5,1,14,tutor,0,firered-leafgreen
6,1,14,machine,0,diamond-pearl
7,1,14,machine,0,platinum
8,1,14,machine,0,heartgold-soulsilver
9,1,14,machine,0,black-white


## `pokemon_species` Table

This table contains **species-level information** about a Pokémon.

### Important columns

- `species_id` → identifies the Pokémon
- `pokemon_category` → species category
- `generation` → generation
- `color`, `shape`, `habitat` → physical/ecological information
- `capture_rate` → capture difficulty
- `base_happiness` → base happiness
- `growth_rate` → growth pattern
- `gender_rate` → gender ratio information
- `hatch_counter` → hatching information
- `egg_group_1`, `egg_group_2` → egg groups
- `is_baby` → whether it is a baby Pokémon
- `is_legendary` → whether it is legendary
- `is_mythical` → whether it is mythical
- `special_status` → derived special classification

The table is connected to `pokemon` using `pokemon_id`.

In [96]:
species_df = pd.read_sql_query("""
    SELECT *
    FROM pokemon_species
    LIMIT 5;
""", conn)

species_df

,species_id,pokemon_category,generation,color,shape,habitat,capture_rate,base_happiness,growth_rate,gender_rate,hatch_counter,egg_group_1,egg_group_2,is_baby,is_legendary,is_mythical,special_status
0,1,Seed Pokémon,generation-i,green,quadruped,grassland,45,70,medium-slow,1,20,monster,plant,0,0,0,Normal
1,2,Seed Pokémon,generation-i,green,quadruped,grassland,45,70,medium-slow,1,20,monster,plant,0,0,0,Normal
2,3,Seed Pokémon,generation-i,green,quadruped,grassland,45,70,medium-slow,1,20,monster,plant,0,0,0,Normal
3,4,Lizard Pokémon,generation-i,red,upright,mountain,45,70,medium-slow,1,20,monster,dragon,0,0,0,Normal
4,5,Flame Pokémon,generation-i,red,upright,mountain,45,70,medium-slow,1,20,monster,dragon,0,0,0,Normal


## How are the tables connected?

The main relationship is through `pokemon_id`.

```text
                         ┌──────────────────┐
                         │     pokemon      │
                         │   pokemon_id     │
                         └────────┬─────────┘
                                  │
             ┌────────────────────┼────────────────────┐                 
             │                    │                    │
             ▼                    ▼                    ▼
   ┌─────────────────┐  ┌──────────────────┐  ┌──────────────────┐
   │ pokemon_types   │  │ pokemon_abilities│  │ pokemon_species  │
   └─────────────────┘  └──────────────────┘  └──────────────────┘
 
```

The important idea is:

**`pokemon` contains the main entity, while the other tables store additional information or relationships related to it.**

This structure prevents unnecessary repetition and keeps the database organized.

## Retrieving Related Pokémon Data

Now let's retrieve information from multiple tables.

We will display:

- Pokémon name
- Type
- Ability

This requires joining the main `pokemon` table with the type and ability tables.

In [97]:
pokemon_details_df = pd.read_sql_query(
    """
    SELECT
        p.name,
        s.pokemon_category,
        t.type_name,
        a.ability_name
    FROM pokemon AS p
    JOIN pokemon_species AS s
        ON p.species_id = s.species_id
    LEFT JOIN pokemon_types AS t
        ON p.pokemon_id = t.pokemon_id
    LEFT JOIN pokemon_abilities AS a
        ON p.pokemon_id = a.pokemon_id
    LIMIT 20;
    """,
    conn
)

pokemon_details_df

,name,pokemon_category,type_name,ability_name
0,bulbasaur,Seed Pokémon,grass,chlorophyll
1,bulbasaur,Seed Pokémon,grass,overgrow
2,bulbasaur,Seed Pokémon,poison,chlorophyll
3,bulbasaur,Seed Pokémon,poison,overgrow
4,ivysaur,Seed Pokémon,grass,chlorophyll
5,ivysaur,Seed Pokémon,grass,overgrow
6,ivysaur,Seed Pokémon,poison,chlorophyll
7,ivysaur,Seed Pokémon,poison,overgrow
8,venusaur,Seed Pokémon,grass,chlorophyll
9,venusaur,Seed Pokémon,grass,overgrow


## Database Design

Our tables also contain database constraints such as:

- **PRIMARY KEY** → uniquely identifies records
- **FOREIGN KEY** → maintains relationships between tables
- **NOT NULL** → prevents required fields from being empty
- **UNIQUE** → prevents duplicate combinations
- **CHECK** → ensures valid values

For example, Pokémon height and weight cannot be negative because we added `CHECK` constraints.

So validation happens at two levels:

**Transformation/Validation layer → Database constraints**

This gives us an additional layer of data quality protection.

In [98]:
conn.close()

print("Database connection closed.")

Database connection closed.


## Summary

Our ETL pipeline finally stores the transformed Pokémon data in SQLite.

The database contains **6 tables**:

1. `pokemon` → main Pokémon information
2. `pokemon_types` → Pokémon types
3. `pokemon_abilities` → Pokémon abilities
4. `pokemon_moves` → Pokémon–move relationships
5. `moves` → move information
6. `pokemon_species` → species information

The database is designed so that related information is stored separately and connected using keys.

This gives us:

**Extract → Transform → Validate → Load → Retrieve**

So the database becomes the **storage layer of our ETL pipeline**, where the processed data can later be queried and used for analysis or applications.